# Лабораторная работа 7. Визуализация текстовых данных (Matplotlib, Seaborn, WordCloud)

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/l07.ipynb)

**Цели занятия:**

- Превращать текст в числа, пригодные для графиков: частоты слов, длины предложений, матрицы «слово × категория»
- Строить частотные диаграммы top-N и проверять закон Ципфа на двойной логарифмической шкале
- Сравнивать категории текстов с помощью boxplot, тепловых карт и облаков слов в subplots
- Связывать оценку (звёзды отзыва) с характеристиками текста

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/OWNER/REPO.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Как визуализировать текст

График строится по числам, поэтому сначала текст превращают в **признаки**: частоты слов, длины предложений и документов, матрицы встречаемости. Перед подсчётом текст нормализуют — иначе `oʻzbek`, `o'zbek` и `o‘zbek` будут посчитаны как три разных слова, и частоты «размажутся».

| Вопрос исследователя | Визуализация | Инструмент |
|---|---|---|
| Какие слова самые частые? | горизонтальная диаграмма top-N | `ax.barh` |
| Подчиняется ли корпус закону Ципфа? | ранг–частота в log-log | `ax.loglog` |
| Различаются ли тексты по длине? | «ящик с усами» по группам | `sns.boxplot` |
| Какие слова характерны для категорий? | тепловая карта | `sns.heatmap` |
| О чём категория? | облако слов | `WordCloud` + `subplots` |
| Связана ли оценка с текстом? | распределение и средние по группам | `sns.countplot`, `sns.barplot` |

## Частотная диаграмма top-N

```python
from collections import Counter
freq = Counter(w for toks in docs["tokens"] for w in toks if w not in stop and len(w) >= 3)
top = freq.most_common(20)
words, counts = zip(*top[::-1])          # переворачиваем: самое частое слово окажется вверху
fig, ax = plt.subplots(figsize=(7, 6))
bars = ax.barh(words, counts)
ax.bar_label(bars, padding=2)             # числа у столбцов
ax.set_title("Топ-20 слов корпуса"); ax.set_xlabel("Частота")
```

Горизонтальные столбцы удобнее вертикальных: длинные слова читаются без поворота.

## Закон Ципфа

Если упорядочить слова по убыванию частоты, то частота слова примерно обратно пропорциональна его **рангу** r:

f(r) ≈ C / r^a, где a ≈ 1.

После логарифмирования: log f = log C − a · log r — это **прямая** с наклоном −a на двойной логарифмической шкале.

```python
import numpy as np
freqs = np.array(sorted(all_freq.values(), reverse=True))
ranks = np.arange(1, len(freqs) + 1)
ax.loglog(ranks, freqs, ".", label="корпус")            # обе оси логарифмические
ax.loglog(ranks, freqs[0] / ranks, "--", label="Ципф: C/r")
slope, intercept = np.polyfit(np.log10(ranks), np.log10(freqs), 1)
```

> **Заметка: Маленький корпус**  
> В 16 коротких новостях много слов встречается 1 раз (**гапаксы**), поэтому «хвост» графика превращается в горизонтальные ступеньки, а наклон получается меньше 1 по модулю. На миллионах слов закон выполняется намного точнее.

## Распределения по группам: boxplot

```python
import seaborn as sns
sns.boxplot(data=sent_df, x="n_words", y="category", order=order)   # горизонтальные ящики
```

Ящики удобно упорядочить по медиане (`order=`), а при большом числе категорий располагать их по оси Y.

## Матрица «слово × категория»: heatmap

```python
long = docs[["category", "tokens"]].explode("tokens")      # одна строка = одно слово
mat = pd.crosstab(long["tokens"], long["category"]).loc[top10]
sns.heatmap(mat, annot=True, fmt="d", cmap="YlGnBu")
```

Если категории сильно различаются по объёму, сравнивайте не абсолютные частоты, а **относительные** (на 1000 слов): `mat / mat.sum() * 1000`.

## Облака слов для нескольких категорий

```python
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, cat in zip(axes.flat, cats):
    wc = WordCloud(width=400, height=250, background_color="white").generate_from_frequencies(f[cat])
    ax.imshow(wc); ax.set_title(cat); ax.axis("off")
```

`axes.flat` перебирает все области сетки по порядку. Не забудьте заменить ʻ ʼ на `'` (встроенный шрифт WordCloud их не содержит).

## Типичные ошибки

> **Внимание: Проверьте перед выводами**  
> - стоп-слова не удалены — в топе одни `va`, `bu`, `bilan`;
> - апострофы не нормализованы — одно слово разбито на несколько;
> - мало данных: категория из 1–2 текстов не позволяет делать статистических выводов;
> - облако слов — иллюстрация, а не измерение: точные значения показывайте столбчатой диаграммой.

## Задание 1. Топ-20 слов: горизонтальная диаграмма

Подготовлены: `docs` — 16 новостей (`file, title, category, …, text, tokens`; `tokens` — нормализованные буквенные слова), `stop` — стоп-слова.

1. `freq` — Counter содержательных слов всего корпуса: не из `stop` и длиной не меньше 3.
2. `top20 = freq.most_common(20)`.
3. Горизонтальная диаграмма: **самое частое слово — вверху**, заголовок и подпись оси X. По желанию подпишите значения (`ax.bar_label`).

<details><summary>💡 Подсказка</summary>

В цикле: `freq.update(w for w in toks if w not in stop and len(w) >= 3)`. Чтобы самое частое слово было вверху, переверните списки: `ax.barh(words[::-1], counts[::-1])`.

</details>

In [ ]:
# Подготовительный код задания
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def normalize(text):
    """Нижний регистр и единые апострофы: oʻ/gʻ (U+02BB), остальные — ʼ (U+02BC)."""
    text = text.lower()
    for ch in ["'", "\u2018", "\u2019", "\x60", "\u02bb"]:
        text = text.replace(ch, "\u02bc")
    return re.sub("([og])\u02bc", "\\1\u02bb", text)

def tokenize(text):
    """Список буквенных слов нормализованного текста."""
    return [w for w in re.findall(r"\w+", normalize(text)) if w.isalpha()]

stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())
docs = pd.read_csv("data/corpus_uz/meta.csv")
docs["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read().split("\n", 1)[1].strip() for f in docs["file"]]
docs["tokens"] = docs["text"].apply(tokenize)

In [ ]:
freq = Counter()
for toks in docs["tokens"]:
    ...

top20 = freq.most_common(20)
print(top20[:5])

words = [w for w, c in top20]
counts = [c for w, c in top20]

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(words, counts, color="steelblue")
plt.show()

## Задание 2. Закон Ципфа: ранг–частота

Проверьте закон Ципфа на корпусе. Здесь учитываются **все** слова, включая стоп-слова.

1. `all_freq` — Counter всех токенов из `docs["tokens"]`.
2. `freqs` — массив numpy частот по убыванию, `ranks` — ранги `1, 2, …, len(freqs)`.
3. `slope` — наклон прямой `np.polyfit(np.log10(ranks), np.log10(freqs), 1)`.
4. График в **двойной логарифмической** шкале: точки корпуса и пунктир идеального закона `freqs[0] / ranks`; заголовок, подписи осей, легенда.

<details><summary>💡 Подсказка</summary>

`all_freq = Counter(w for toks in docs["tokens"] for w in toks)`; `freqs = np.array(sorted(all_freq.values(), reverse=True))`; `ranks = np.arange(1, len(freqs) + 1)`; `ax.loglog(...)` или `ax.set_xscale("log")`, `ax.set_yscale("log")`.

</details>

In [ ]:
# Подготовительный код задания
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def normalize(text):
    """Нижний регистр и единые апострофы: oʻ/gʻ (U+02BB), остальные — ʼ (U+02BC)."""
    text = text.lower()
    for ch in ["'", "\u2018", "\u2019", "\x60", "\u02bb"]:
        text = text.replace(ch, "\u02bc")
    return re.sub("([og])\u02bc", "\\1\u02bb", text)

def tokenize(text):
    """Список буквенных слов нормализованного текста."""
    return [w for w in re.findall(r"\w+", normalize(text)) if w.isalpha()]

stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())
docs = pd.read_csv("data/corpus_uz/meta.csv")
docs["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read().split("\n", 1)[1].strip() for f in docs["file"]]
docs["tokens"] = docs["text"].apply(tokenize)

In [ ]:
all_freq = ...
freqs = ...
ranks = ...
slope, intercept = ...

print("Разных слов:", len(freqs), "наклон:", slope)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(ranks, freqs, ".", label="корпус")
plt.show()

## Задание 3. Длина предложений по категориям (boxplot)

Сравните длину предложений в разных категориях новостей.

1. Разбейте `text` каждой статьи на предложения: `re.split(r"(?<=[.!?])\s+", text)`. Для каждого предложения посчитайте число слов `len(tokenize(s))` (пустые пропустите).
2. `sent_df` — DataFrame со столбцами `category` и `n_words` (одна строка = одно предложение).
3. `med` — медиана `n_words` по категориям, по убыванию.
4. `sns.boxplot` с горизонтальными ящиками (категории по оси Y) в порядке `med.index`, заголовок и подпись оси X.

<details><summary>💡 Подсказка</summary>

В цикле: `n = len(tokenize(s))`, и если `n > 0`: `rows.append({"category": cat, "n_words": n})`. Медиана: `sent_df.groupby("category")["n_words"].median().sort_values(ascending=False)`. График: `sns.boxplot(data=sent_df, x="n_words", y="category", order=med.index, ax=ax)`.

</details>

In [ ]:
# Подготовительный код задания
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def normalize(text):
    """Нижний регистр и единые апострофы: oʻ/gʻ (U+02BB), остальные — ʼ (U+02BC)."""
    text = text.lower()
    for ch in ["'", "\u2018", "\u2019", "\x60", "\u02bb"]:
        text = text.replace(ch, "\u02bc")
    return re.sub("([og])\u02bc", "\\1\u02bb", text)

def tokenize(text):
    """Список буквенных слов нормализованного текста."""
    return [w for w in re.findall(r"\w+", normalize(text)) if w.isalpha()]

stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())
docs = pd.read_csv("data/corpus_uz/meta.csv")
docs["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read().split("\n", 1)[1].strip() for f in docs["file"]]
docs["tokens"] = docs["text"].apply(tokenize)
import seaborn as sns

In [ ]:
rows = []
for cat, text in zip(docs["category"], docs["text"]):
    for s in re.split(r"(?<=[.!?])\s+", text):
        ...

sent_df = pd.DataFrame(rows)
med = ...

fig, ax = plt.subplots(figsize=(8, 6))
# sns.boxplot(...)
plt.show()

## Задание 4. Тепловая карта «слово × категория»

`freq` — Counter содержательных слов корпуса (как в задании 1).

1. `top10` — список 10 самых частых слов из `freq`.
2. `mat` — DataFrame: строки — слова `top10`, столбцы — все 12 категорий, значения — сколько раз слово встречается в статьях категории (целые числа).
3. Тепловая карта `sns.heatmap` с подписями значений (`annot=True, fmt="d"`) и заголовком.

Какие слова «привязаны» к одной категории, а какие встречаются везде?

<details><summary>💡 Подсказка</summary>

`top10 = [w for w, _ in freq.most_common(10)]`; внутри цикла: `for w in top10: mat.loc[w, cat] += c[w]`. Другой путь: `long = docs[["category", "tokens"]].explode("tokens")`, `pd.crosstab(long["tokens"], long["category"]).loc[top10]`.

</details>

In [ ]:
# Подготовительный код задания
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def normalize(text):
    """Нижний регистр и единые апострофы: oʻ/gʻ (U+02BB), остальные — ʼ (U+02BC)."""
    text = text.lower()
    for ch in ["'", "\u2018", "\u2019", "\x60", "\u02bb"]:
        text = text.replace(ch, "\u02bc")
    return re.sub("([og])\u02bc", "\\1\u02bb", text)

def tokenize(text):
    """Список буквенных слов нормализованного текста."""
    return [w for w in re.findall(r"\w+", normalize(text)) if w.isalpha()]

stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())
docs = pd.read_csv("data/corpus_uz/meta.csv")
docs["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read().split("\n", 1)[1].strip() for f in docs["file"]]
docs["tokens"] = docs["text"].apply(tokenize)

freq = Counter(w for toks in docs["tokens"] for w in toks if w not in stop and len(w) >= 3)
import seaborn as sns

In [ ]:
top10 = ...
cats = sorted(docs["category"].unique())

mat = pd.DataFrame(0, index=top10, columns=cats)
for cat, toks in zip(docs["category"], docs["tokens"]):
    c = Counter(toks)
    ...

fig, ax = plt.subplots(figsize=(12, 5))
# sns.heatmap(...)
plt.show()

## Задание 5. Облака слов по категориям

Постройте облака слов для четырёх категорий, в которых по две статьи: `cats4 = ["jamiyat", "taʼlim", "texnologiya", "turizm"]`.

1. Для каждой категории посчитайте частоты содержательных слов (не из `stop`, длина ≥ 3) и замените в словах ʻ и ʼ на `'`.
2. Постройте `WordCloud(width=400, height=250, background_color="white", max_words=40)` по этим частотам.
3. Сохраните облака в словарь `clouds` (`{категория: WordCloud}`) и покажите их на рисунке 2×2: в каждой области — облако, заголовок = название категории, оси скрыты.

<details><summary>💡 Подсказка</summary>

В цикле по статьям: `f.update(w.replace("ʻ", "'").replace("ʼ", "'") for w in toks if w not in stop and len(w) >= 3)`. Затем `clouds[cat] = WordCloud(...).generate_from_frequencies(f)`, `ax.imshow(clouds[cat])`.

</details>

In [ ]:
# Подготовительный код задания
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def normalize(text):
    """Нижний регистр и единые апострофы: oʻ/gʻ (U+02BB), остальные — ʼ (U+02BC)."""
    text = text.lower()
    for ch in ["'", "\u2018", "\u2019", "\x60", "\u02bb"]:
        text = text.replace(ch, "\u02bc")
    return re.sub("([og])\u02bc", "\\1\u02bb", text)

def tokenize(text):
    """Список буквенных слов нормализованного текста."""
    return [w for w in re.findall(r"\w+", normalize(text)) if w.isalpha()]

stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())
docs = pd.read_csv("data/corpus_uz/meta.csv")
docs["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read().split("\n", 1)[1].strip() for f in docs["file"]]
docs["tokens"] = docs["text"].apply(tokenize)
from wordcloud import WordCloud

In [ ]:
cats4 = ["jamiyat", "taʼlim", "texnologiya", "turizm"]
clouds = {}

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, cat in zip(axes.flat, cats4):
    f = Counter()
    for toks in docs.loc[docs["category"] == cat, "tokens"]:
        ...
    # clouds[cat] = WordCloud(...).generate_from_frequencies(f)
    # ax.imshow(...); ax.set_title(cat); ax.axis("off")

plt.tight_layout()
plt.show()

## Задание 6. Отзывы: звёзды и длина текста

Файл `data/reviews_uz.csv` — 180 отзывов о книгах (`stars` 1–5, `text` на узбекском).

1. Прочитайте его в `rev` и добавьте столбец `n_words` — число слов в тексте (`str.split`).
2. `stars_counts` — число отзывов с каждой оценкой (по возрастанию звёзд).
3. `len_by_stars` — средняя длина отзыва в словах для каждой оценки.
4. Рисунок 1×2: слева — распределение оценок (`sns.countplot` или `bar`), справа — средняя длина по оценкам (`sns.barplot`). Заголовки и подписи осей обязательны.

Пишут ли недовольные покупатели длиннее довольных?

<details><summary>💡 Подсказка</summary>

`rev["n_words"] = rev["text"].str.split().str.len()`; `stars_counts = rev["stars"].value_counts().sort_index()`; `len_by_stars = rev.groupby("stars")["n_words"].mean()`; `sns.countplot(data=rev, x="stars", ax=axes[0])`, `sns.barplot(data=rev, x="stars", y="n_words", ax=axes[1])`.

</details>

In [ ]:
# Подготовительный код задания
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
rev = pd.read_csv("data/reviews_uz.csv")
rev["n_words"] = ...

stars_counts = ...
len_by_stars = ...
print(stars_counts)
print(len_by_stars)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# слева: распределение оценок
# справа: средняя длина по оценкам
plt.tight_layout()
plt.show()

## Самостоятельная работа

## Самостоятельная работа

1. Постройте график Ципфа для отзывов `reviews_uz.csv` и сравните наклон с корпусом новостей. Почему он отличается? (Подсказка: отзывы собраны из повторяющихся шаблонных фраз.)
2. Для корпуса новостей постройте тепловую карту **относительных** частот (на 1000 слов категории) и сравните её с абсолютной.
3. Тема для доклада: «Анализ больших объёмов данных, формируемых в социальных сетях» — какие визуализации подходят для потока постов (`data/tweets_stream.csv`): динамика по минутам, хештеги, облако слов, тепловая карта «платформа × язык».